In [ ]:
# Install quantized training/inference libs (bitsandbytes) and TRL; -q = quiet
!pip install -q --upgrade bitsandbytes trl
# Download the course evaluator script (runs model on test set, plots error vs actual price)
!wget -q https://raw.githubusercontent.com/ed-donner/llm_engineering/main/week7/util.py -O util.py

In [ ]:
# --- Imports ---

# Colab: read secrets (e.g. HF token) from notebook "Secrets" panel
from google.colab import userdata
# HuggingFace Hub login for gated models and pushing adapters
from huggingface_hub import login
import torch
import transformers
# Base model + tokenizer; BitsAndBytesConfig = 4bit/8bit quantization; set_seed for reproducibility
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
# Load dataset from Hub (items_prompts_lite / items_prompts_full)
from datasets import load_dataset, Dataset, DatasetDict
from datetime import datetime
# PEFT = Parameter-Efficient Fine-Tuning; load adapter on top of base model
from peft import PeftModel
# Evaluator: runs predictor on test set, parses price, plots error and R²
from util import evaluate

In [ ]:
# --- Constants ---
# Base causal LM to load (before applying PEFT adapter)
BASE_MODEL = "meta-llama/Llama-3.2-3B"
# Project name used in Hub run IDs
PROJECT_NAME = "price"
# HuggingFace username that owns the fine-tuned adapter
HF_USER = "ed-donner"

# True = use small "lite" dataset and lite run; False = full dataset and full run
LITE_MODE = True

# Dataset on Hub: lite (small) or full item prompts with prices
DATA_USER = "ed-donner"
DATASET_NAME = f"{DATA_USER}/items_prompts_lite" if LITE_MODE else f"{DATA_USER}/items_prompts_full"

# Which training run's adapter to load (run name + optional git revision)
if LITE_MODE:
  RUN_NAME = "2025-11-30_15.10.55-lite"
  REVISION = None
else:
  RUN_NAME = "2025-11-28_18.47.07"
  REVISION = "b19c8bfea3b6ff62237fbb0a8da9779fc12cefbd"

# Full Hub model id: username/model-run-name
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

# --- QLoRA hyper-parameters ---
# True = 4-bit quantization (smaller, slightly less accurate); False = 8-bit
QUANT_4_BIT = True
# GPU compute capability; major version >= 8 (e.g. Ampere) can use bfloat16
capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8

In [ ]:
# --- Log in to HuggingFace ---
# Read token from Colab Secrets (add HF_TOKEN in the key list)
hf_token = userdata.get('HF_TOKEN')
# Authenticate for this session and optionally store in git credential helper
login(hf_token, add_to_git_credential=True)

In [ ]:
# Load the dataset from HuggingFace Hub (prompts + completion prices)
dataset = load_dataset(DATASET_NAME)
# Use the test split for evaluation (predictor will get 'prompt' and we compare to 'completion')
test = dataset['test']

In [ ]:
# --- Pick quantization config for loading the base model (QLoRA) ---
if QUANT_4_BIT:
  quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,   # Extra quantization of the 4-bit scales (saves more memory)
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,  # Dtype for matmul during forward
    bnb_4bit_quant_type="nf4"          # 4-bit normal float type
  )
else:
  quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
  )

In [ ]:
# --- Load tokenizer and model ---
# Tokenizer for the base model (needed for input encoding and decoding)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
# Llama has no pad token by default; use EOS so we can pad batches
tokenizer.pad_token = tokenizer.eos_token
# Pad on the right so left-aligned prompt is unchanged for generation
tokenizer.padding_side = "right"

# Base model loaded in 4bit/8bit to fit on GPU; device_map="auto" spreads over available devices
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
# Tell generation to use the same pad token id (avoids warnings)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

# Load the fine-tuned LoRA/QLoRA adapter from the Hub on top of the base model
if REVISION:
  fine_tuned_model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME, revision=REVISION)
else:
  fine_tuned_model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME)

# Print total model memory (base + adapter)
print(f"Memory footprint: {fine_tuned_model.get_memory_footprint() / 1e6:.1f} MB")

In [ ]:
def model_predict(item):
    # Encode the item prompt (e.g. "Title: ...") to token ids and move to GPU
    inputs = tokenizer(item["prompt"], return_tensors="pt").to("cuda")
    # No gradient needed for inference; generate up to 8 new tokens (enough for a price)
    with torch.no_grad():
        output_ids = fine_tuned_model.generate(**inputs, max_new_tokens=8)
    # Length of the prompt in tokens so we slice off only the generated part
    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0, prompt_len:]
    # Decode the generated token ids to a string (e.g. "$12.99" or "42.00")
    return tokenizer.decode(generated_ids)

In [ ]:
# Fix random seed so runs are reproducible (e.g. sampling if used in generate)
set_seed(42)
# Run model on test set, parse predicted price, compare to ground truth, plot error and R²
evaluate(model_predict, test)